[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/templates/27_vit_patch.ipynb)

# 🟡 Medium: Vision Transformer Patch Embedding

*Attention & Transformers*
Implement the **patch embedding** that turns an image into a sequence of tokens
for a Vision Transformer.

$$(B, C, H, W) \;\longrightarrow\; (B, N, D), \qquad
N = \frac{H}{p}\cdot\frac{W}{p}$$

### Signature
```python
class PatchEmbedding(nnx.Module):
    def __init__(self, img_size, patch_size, in_channels, embed_dim,
                 *, rngs: nnx.Rngs): ...
    def __call__(self, x): ...
```

### Requirements
- Input is **NCHW** — `(B, C, H, W)`, the PyTorch layout
- `self.patch_size`, `self.num_patches = (img_size // patch_size) ** 2`
- `self.proj`: `nnx.Linear(in_channels * patch_size * patch_size, embed_dim)`
- Patches in **row-major** order, and each flattened as `(C, p, p)`

`nnx.Linear` is an allowed building block; the exercise is the reshape.

### The reshape, step by step
```
(B, C, H, W)                      split each spatial axis
  -> (B, C, n_h, p, n_w, p)       grid index and within-patch index
  -> (B, n_h, n_w, C, p, p)       grid axes first
  -> (B, n_h*n_w, C*p*p)          flatten to tokens
```

### Why this is exactly a strided convolution
A `Conv2d(C, D, kernel_size=p, stride=p)` computes the identical thing: each
output position sees one non-overlapping `p × p` patch and projects it to `D`
channels. Real ViT implementations use the conv because it is one fused kernel,
but the reshape makes it obvious that **no spatial mixing happens here** — every
patch is embedded independently, and all interaction between patches is left to
the attention layers.

### What the reshape throws away, and what gets it back
After this step the model has no idea where any patch came from: the sequence
is permutation-equivariant, so shuffling the patches shuffles the outputs
identically. That is why ViT adds **position embeddings** immediately
afterwards, and why it needs far more data than a CNN — translation
equivariance is baked into a conv, but a transformer has to learn it.

### The trap
The wrong transpose gives the right output *shape*. `(B, n_h*n_w, C*p*p)` comes
out either way, so shape assertions pass while the patch contents are scrambled.
The tests below check values against a hand-built patch.

In [ ]:
# Colab setup (no-op when running locally).
# jax-judge is not published on PyPI, so the judge is installed from the
# repo itself. Regenerate with JAXCODE_REPO=you/YourFork to point this at
# your own fork:  JAXCODE_REPO=you/JAXCode make notebooks
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q flax optax')
    get_ipython().run_line_magic(
        'pip', 'install -q git+https://github.com/YOUR-GITHUB-USERNAME/JAXCode.git')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp
from flax import nnx

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

import jax
import jax.numpy as jnp
from flax import nnx


class PatchEmbedding(nnx.Module):
    """Image (B, C, H, W) -> patch tokens (B, N, embed_dim)."""

    def __init__(self, img_size: int, patch_size: int, in_channels: int,
                 embed_dim: int, *, rngs: nnx.Rngs):
        pass  # Replace this

    def __call__(self, x):
        """(B, C, H, W) -> (B, num_patches, embed_dim)"""
        pass  # Replace this

In [ ]:
# 🔍 Scratch cell — poke at your implementation
import jax
import jax.numpy as jnp
from flax import nnx

pe = PatchEmbedding(img_size=32, patch_size=8, in_channels=3, embed_dim=64,
                    rngs=nnx.Rngs(params=0))
x = jax.random.normal(jax.random.key(1), (2, 3, 32, 32))    # NCHW

print("image :", x.shape)
print("tokens:", pe(x).shape, f"(num_patches={pe.num_patches})")
print("patch feature dim:", 3 * 8 * 8, "-> projected to 64")

In [ ]:
# ✅ SUBMIT — run this cell to check your solution
from jax_judge import check, hint, solution, status

check("vit_patch")

# hint("vit_patch")      # stuck? nudge without the answer
# solution("vit_patch")  # spoiler: the reference implementation
# status()               # your dashboard across all problems